# Preprocessing
This notebook generates training/validation/test chips and both YOLO & COCO labels from the xView dataset (geojson + geoTIFFs), with stats and random visualization.

In [1]:
# Standard Library
import os
import time
import random
import json
import yaml
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict, Counter
import concurrent.futures
import multiprocessing as mp
import re

# Data Manipulation and Processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patches as mpatches
from PIL import Image, ImageOps

# Computer Vision and Image Processing
import cv2
import imageio.v3 as imageio

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import AlexNet_Weights
from torchvision.transforms import functional as TF

# Machine Learning
from sklearn.svm import SVC

# Progress Monitoring
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook

## Setup

In [2]:
# Data source
DATA_FOLDER_NAME = "data"
INPUT_DATASET_NAME = "xview-dataset"
IMAGE_FOLDER_NAME = "train_images"
INPUT_LABELS_FOLDER_NAME = "train_labels"
LABELS_XML_NAME = "xView_train.geojson"

In [3]:
# Output folders and file names
OUTPUT_DATASET_NAME = "xview-yolo-dataset"
CLASS_MAP_JSON_NAME = "xView_class_map.json"
OUTPUT_COCO_JSON_NAME = "coco_annotations.json"
OUTPUT_IMAGE_FOLDER_NAME = "images"
OUTPUT_CONFIG_FOLDER_NAME = "YOLO_config"
OUTPUT_DATAFRAME_NAME = "xview_labels.parquet"

In [4]:
# Parameters
CHUNK_WIDTH = 320
CHUNK_HEIGHT = 320
MIN_CHUNK_HEIGHT = 320
MIN_CHUNK_WIDTH = 320
IMAGE_WRITING = True
JPEG_COMPRESSION = 95
RANDOM_SEED = 42
DEBUG = False

In [5]:
# Define paths

input_dataset_path = Path("../data/raw/xview-dataset")
output_dataset_path = Path("../data/processed/")
future_dataset_image_folder = Path(f"../data/processed/{OUTPUT_IMAGE_FOLDER_NAME}")
future_dataset_config_folder = Path(f"../data/processed/{OUTPUT_CONFIG_FOLDER_NAME}")

labels_json_path = input_dataset_path / INPUT_LABELS_FOLDER_NAME / LABELS_XML_NAME
image_folder_path = input_dataset_path /IMAGE_FOLDER_NAME
save_images_folder_path = output_dataset_path / OUTPUT_IMAGE_FOLDER_NAME
output_data_parquet_path = output_dataset_path / OUTPUT_DATAFRAME_NAME
output_json_map_path = output_dataset_path / CLASS_MAP_JSON_NAME
class_map_path = output_dataset_path / CLASS_MAP_JSON_NAME
config_folder_path = output_dataset_path / OUTPUT_CONFIG_FOLDER_NAME
coco_json_path = output_dataset_path / OUTPUT_COCO_JSON_NAME

print (f"Input dataset path: {input_dataset_path}")
print (f"Output dataset path: {output_dataset_path}")
print (f"Future dataset image folder: {future_dataset_image_folder}")
print (f"Future dataset config folder: {future_dataset_config_folder}")
print (f"Labels JSON path: {labels_json_path}")
print (f"Image folder path: {image_folder_path}")
print (f"Save images folder path: {save_images_folder_path}")
print (f"Output data parquet path: {output_data_parquet_path}")
print (f"Output JSON map path: {output_json_map_path}")
print (f"Class map path: {class_map_path}")
print (f"Config folder path: {config_folder_path}")
print (f"COCO JSON path: {coco_json_path}")

Input dataset path: ..\data\raw\xview-dataset
Output dataset path: ..\data\processed
Future dataset image folder: ..\data\processed\images
Future dataset config folder: ..\data\processed\YOLO_config
Labels JSON path: ..\data\raw\xview-dataset\train_labels\xView_train.geojson
Image folder path: ..\data\raw\xview-dataset\train_images
Save images folder path: ..\data\processed\images
Output data parquet path: ..\data\processed\xview_labels.parquet
Output JSON map path: ..\data\processed\xView_class_map.json
Class map path: ..\data\processed\xView_class_map.json
Config folder path: ..\data\processed\YOLO_config
COCO JSON path: ..\data\processed\coco_annotations.json


In [6]:
def make_empty_dir(directory):
    if directory.is_dir():
        shutil.rmtree(directory)
    os.makedirs(directory)

make_empty_dir(config_folder_path)

if IMAGE_WRITING:
    make_empty_dir(save_images_folder_path)

random.seed(RANDOM_SEED)

print(f'The input images are found at {image_folder_path}')
print(f'The input labels are found at  {labels_json_path}')
print(f'Configuration files will be saved to {config_folder_path}')
print(f'YOLO image files will be saved to {save_images_folder_path}')

The input images are found at ..\data\raw\xview-dataset\train_images
The input labels are found at  ..\data\raw\xview-dataset\train_labels\xView_train.geojson
Configuration files will be saved to ..\data\processed\YOLO_config
YOLO image files will be saved to ..\data\processed\images


In [7]:
from pathlib import Path

def check_path(p, name, expect_file=False):
    print(f"\n{name}")
    print("  raw :", p)
    print("  abs :", p.resolve())
    print("  exists :", p.exists())
    if p.exists():
        if expect_file:
            print("  type : file")
            print("  size :", p.stat().st_size, "bytes")
        else:
            print("  type : dir")
            print("  sample :", [x.name for x in list(p.glob('*'))[:5]])

print("\n========== PATH CHECK ==========")

check_path(input_dataset_path, "input_dataset_path")
check_path(labels_json_path, "labels_json_path", expect_file=True)
check_path(image_folder_path, "image_folder_path")
check_path(output_dataset_path, "output_dataset_path")
check_path(save_images_folder_path, "save_images_folder_path")
check_path(config_folder_path, "config_folder_path")
check_path(output_data_parquet_path, "output_data_parquet_path", expect_file=True)
check_path(output_json_map_path, "output_json_map_path", expect_file=True)


========== PATH CHECK ==========

input_dataset_path
  raw : ..\data\raw\xview-dataset
  abs : C:\Users\msagh\Documents\PersonalProjects\xView_detect\data\raw\xview-dataset
  exists : True
  type : dir
  sample : ['train_images', 'train_labels', 'val_images']

labels_json_path
  raw : ..\data\raw\xview-dataset\train_labels\xView_train.geojson
  abs : C:\Users\msagh\Documents\PersonalProjects\xView_detect\data\raw\xview-dataset\train_labels\xView_train.geojson
  exists : True
  type : file
  size : 372241031 bytes

image_folder_path
  raw : ..\data\raw\xview-dataset\train_images
  abs : C:\Users\msagh\Documents\PersonalProjects\xView_detect\data\raw\xview-dataset\train_images
  exists : True
  type : dir
  sample : ['10.tif', '100.tif', '102.tif', '1036.tif', '1037.tif']

output_dataset_path
  raw : ..\data\processed
  abs : C:\Users\msagh\Documents\PersonalProjects\xView_detect\data\processed
  exists : True
  type : dir
  sample : ['images', 'proposals', 'YOLO_config']

save_images_f

## Helper functions

In [8]:
# Define a function to extract bounding boxes from the DataFrame

def get_boxes(in_df, class_list = []):
    if class_list:
        in_df = in_df[in_df["TYPE_ID"].isin(class_list)]    # Filter the DataFrame to include only the specified classes
    unique_images = in_df.IMAGE_ID.unique().tolist()        # Get the unique image IDs from the DataFrame
    boxes = {}

    for image in tqdm_notebook(unique_images):
        mask = in_df["IMAGE_ID"] == image                                   # Select only the rows related to a specific image
        masked = in_df[mask][["TYPE_ID", "X_MIN", "Y_MIN", "X_MAX", "Y_MAX"]]   # Extract only the required columns
        boxes[image] = masked.values.tolist()                               # Convert the values to a list and save them in the dictionary
    return boxes

In [9]:
# Define a function to print the first n lines of a file

def print_first_n_lines(file_path, n):
    try:
        with open(file_path, 'r') as file:
            for line_num, line in enumerate(file, 1):
                if line_num > n:                  
                    break
                print(line.strip()) 
    except FileNotFoundError:
        print('Unable to open file')

In [10]:
# Define a function to load an image to visualize it

def load_image(file_pth):
    image_obj = cv2.imread(file_pth)
    if image_obj is None:
        print(f'Unable to load image at {file_pth}')
        return None                        
    image_obj = cv2.cvtColor(image_obj, cv2.COLOR_BGR2RGB)  
    return image_obj

In [11]:
# Define a function to load an image for processing

def load_bgr_image(file_pth):
    image_obj = cv2.imread(file_pth)
    if image_obj is None:
        print(f'Unable to load image at {file_pth}')
        return None
    return image_obj

In [12]:
# Define a function to display images with bounding boxes

def display_images(image_list, boxes_dictionary, image_folder, max_images=6, no_cols=1, text=False, class_map={}):
    total_images = len(image_list)
    display_images = min(max_images, total_images)
    no_rows = display_images // no_cols + (display_images % no_cols > 0)             # Calculate number of rows needed
    fig, axs = plt.subplots(no_rows, no_cols, figsize=(10, 10*no_rows/no_cols*3/4))  # Create the grid layout
    axs = axs.flatten()                                                              # Flatten the array of axes

    for k, image_name in enumerate(image_list[:display_images]):
        image_path = str(image_folder / image_name)  
        image = load_image(image_path)  

        # Design the bounding boxes
        if image_name in boxes_dictionary:
            for box in boxes_dictionary[image_name]:
                box_id, x_min, y_min, x_max, y_max = box
                x_min, y_min, x_max, y_max = int(x_min), int(y_max), int(x_max), int(y_min)
                cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0,255,0), 3)      # Draw the rectangle
                if text:
                    if class_map:
                        box_label = class_map[box_id]   # Map the class
                    else:
                        box_label = str(box_id)
                    cv2.putText(image, box_label, (x_min, y_max-10), cv2.FONT_HERSHEY_SIMPLEX, 2, (36,255,12), 4)  # Write the text

        axs[k].set_title(f"Image {image_name}", fontsize=12)
        axs[k].imshow(image)
        axs[k].set_axis_off()

    plt.tight_layout()
    plt.show()
    return

In [13]:
# Convert YOLO coordinates to OpenCV coordinates ((left, top), (right, bottom))

def get_corners(x_center, y_center, annotation_width, annotation_height, image_width, image_height):
    x_center, y_center, annotation_width, annotation_height = float(x_center), float(y_center), float(annotation_width), float(annotation_height)
    # Calculate the corners of the bounding box
    left = (x_center - annotation_width/2)*image_width            
    top = (y_center - annotation_height/2)*image_height           
    right = (x_center + annotation_width/2)*image_width           
    bottom = (y_center + annotation_height/2)*image_height        
    return int(left), int(top), int(right), int(bottom)

In [14]:
# Show images with bounding boxes defined in YOLO format    

def display_yolo_images(image_list, image_folder, max_images=6, no_cols=1, text=False, class_map={}):
    total_images = len(image_list)
    display_images = min(max_images, total_images)
    no_rows = display_images // no_cols + (display_images % no_cols > 0)
    _, axs = plt.subplots(no_rows, no_cols, figsize=(10, 10*no_rows/no_cols*3/4))
    axs = axs.flatten()

    for k, image_name in enumerate(image_list[:display_images]):
        image_path = image_folder / image_name
        text_filename = image_path.stem + '.txt'     # Build the name of the bounding box file
        boxes_path = image_folder / text_filename
        image = load_image(str(image_path))  
        image_height, image_width, _ = image.shape  
        with open(boxes_path) as text_file:
            annotations = [line.rstrip().split() for line in text_file]  

        # Draw the bounding boxes
        for annotation in annotations:
            class_id = annotation[0]
            x_centre, y_centre, width, height = annotation[1], annotation[2], annotation[3], annotation[4]
            x_min, y_min, x_max, y_max = get_corners(x_centre, y_centre, width, height, image_width, image_height)
            cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0,255,0), 3)
            if text:
                if class_map:
                    box_label = class_map[int(class_id)]
                else:
                    box_label = str(class_id)
                cv2.putText(image, box_label, (x_min, y_max-10), cv2.FONT_HERSHEY_SIMPLEX, 2, (36,255,12), 4)

        axs[k].set_title(f"Image {image_name}", fontsize=12)
        axs[k].imshow(image)
        axs[k].set_axis_off()

    plt.tight_layout()
    plt.show()
    return

In [15]:
# Find bounding boxes contained in a section of the image and return them in YOLO format

def match_boxes(box_list, chunk_limits):
    boxes_lists = []
    chunk_left, chunk_top = chunk_limits[0], chunk_limits[1]                    # Chunk limits (left, top)
    width, height = chunk_limits[2], chunk_limits[3]                      # Chunk width and height
    for box in box_list:
        original_left, original_top, original_right, original_bottom = box[1], box[2], box[3], box[4] # Original bounding box coordinates
        left, right = (original_left - chunk_left)/width, (original_right - chunk_left)/width    # Normalize with respect to chunk limits
        top, bottom = (original_top - chunk_top)/height, (original_bottom - chunk_top)/height
    
        # Check if the bounding box is contained in the section
        horizontal_match = (0 <= left < 1) or (0 < right <= 1) # Horizontal match
        vertical_match = (0 <= top < 1) or (0 < bottom <= 1) # Vertical match

        if vertical_match and horizontal_match:
            clipped = np.clip([left, top, right, bottom], a_min=0, a_max=1)  # Clip values between 0 and 1
            left, top, right, bottom = clipped[0], clipped[1], clipped[2], clipped[3]
            bounding_box = [str(box[0]),
                            str(round((left + right)/2, 5)),
                            str(round((top + bottom)/2, 5)),
                            str(round(right - left, 5)),
                            str(round(bottom - top, 5))]       # Format in YOLO
            boxes_lists.append(bounding_box)
    return boxes_lists

## Data extraction and cleaning

In [16]:
# Open a json file containing annotations and load the data
with open(labels_json_path, 'r') as infile:
    data = json.load(infile)                        # Load the json file in a dictionary
    keys = list(data.keys())                        # Extract the keys from the dictionary

# Extract the list of annotations (features) from the JSON
feature_list = data['features']

# Define the columns for the DataFrame
COLUMNS = ['IMAGE_ID', 'TYPE_ID', 'X_MIN', 'Y_MIN', 'X_MAX', 'Y_MAX', 'LONG', 'LAT']

data = []
for feature in tqdm_notebook(feature_list):  
    properties = feature['properties']               # Extract the properties of the feature (dictionary)
    image_id = properties['image_id']                  # Image identifier
    type_id = properties['type_id']                  # Type of annotated object
    bbox = properties['bounds_imcoords'].split(",")  # Bounding box coordinates

    geometry = feature['geometry']  
    coordinates = geometry['coordinates'][0]              # List of coordinates (list of lists)

    # Calculate the geographic center point
    longitude = coordinates[0][0] / 2 + coordinates[2][0] / 2       # Average of the longitudes of opposite vertices
    latitude = coordinates[0][1] / 2 + coordinates[1][1] / 2        # Average of the latitudes of opposite vertices

    # Create a row with all relevant information
    one_row = [image_id, type_id, bbox[0], bbox[1], bbox[2], bbox[3], longitude, latitude]
    data.append(one_row)  

# Count the total number of instances in the dataset
instances = len(data)
print(f'There are {instances} instances of objects in the original dataset')

  0%|          | 0/601937 [00:00<?, ?it/s]

There are 601937 instances of objects in the original dataset


In [17]:
# Extract the columns of interest

df = pd.DataFrame(data, columns = COLUMNS)
df[['X_MIN', 'Y_MIN', 'X_MAX', 'Y_MAX']] = df[['X_MIN', 'Y_MIN', 'X_MAX', 'Y_MAX']].apply(pd.to_numeric)
df.head()

,IMAGE_ID,TYPE_ID,X_MIN,Y_MIN,X_MAX,Y_MAX,LONG,LAT
0,2355.tif,73,2712,1145,2746,1177,-90.531640,14.566091
1,2355.tif,73,2720,2233,2760,2288,-90.531603,14.562313
2,2355.tif,73,2687,1338,2740,1399,-90.531694,14.565379
3,2355.tif,73,2691,1201,2730,1268,-90.531704,14.565838
4,2355.tif,73,2671,838,2714,869,-90.531766,14.567149


In [18]:
# Navigation in the dataset showed that some annotations are wrong
# Let's remove them


df = df[(df.TYPE_ID != 75) & (df.TYPE_ID != 82)]   # removing erroneous labels
print(f'{instances - len(df)} rows removed, leaving {len(df)} rows')
df.head()

79 rows removed, leaving 601858 rows


,IMAGE_ID,TYPE_ID,X_MIN,Y_MIN,X_MAX,Y_MAX,LONG,LAT
0,2355.tif,73,2712,1145,2746,1177,-90.531640,14.566091
1,2355.tif,73,2720,2233,2760,2288,-90.531603,14.562313
2,2355.tif,73,2687,1338,2740,1399,-90.531694,14.565379
3,2355.tif,73,2691,1201,2730,1268,-90.531704,14.565838
4,2355.tif,73,2671,838,2714,869,-90.531766,14.567149


In [19]:
# Also, image 1395 is missing, so let's remove the annotations related to it

old_length = len(df)
df = df[df.IMAGE_ID != '1395.tif']
print(f'{old_length - len(df)} rows removed, leaving {len(df)}')
df.head()

131 rows removed, leaving 601727


,IMAGE_ID,TYPE_ID,X_MIN,Y_MIN,X_MAX,Y_MAX,LONG,LAT
0,2355.tif,73,2712,1145,2746,1177,-90.531640,14.566091
1,2355.tif,73,2720,2233,2760,2288,-90.531603,14.562313
2,2355.tif,73,2687,1338,2740,1399,-90.531694,14.565379
3,2355.tif,73,2691,1201,2730,1268,-90.531704,14.565838
4,2355.tif,73,2671,838,2714,869,-90.531766,14.567149


In [20]:
# It is also useful to convert the IDs of the classes to a more compact format, starting from 0 and without missing numbers. Let's do that and save the mapping in a JSON file.

old_dict = {
    11:'Fixed-wing Aircraft', 12:'Small Aircraft', 13:'Passenger/Cargo Plane', 15:'Helicopter',
    17:'Passenger Vehicle', 18:'Small Car', 19:'Bus', 20:'Pickup Truck', 21:'Utility Truck',
    23:'Truck', 24:'Cargo Truck', 25:'Truck Tractor w/ Box Trailer', 26:'Truck Tractor',27:'Trailer',
    28:'Truck Tractor w/ Flatbed Trailer', 29:'Truck Tractor w/ Liquid Tank', 32:'Crane Truck',
    33:'Railway Vehicle', 34:'Passenger Car', 35:'Cargo/Container Car', 36:'Flat Car', 37:'Tank car',
    38:'Locomotive', 40:'Maritime Vessel', 41:'Motorboat', 42:'Sailboat', 44:'Tugboat', 45:'Barge',
    47:'Fishing Vessel', 49:'Ferry', 50:'Yacht', 51:'Container Ship', 52:'Oil Tanker',
    53:'Engineering Vehicle', 54:'Tower crane', 55:'Container Crane', 56:'Reach Stacker',
    57:'Straddle Carrier', 59:'Mobile Crane', 60:'Dump Truck', 61:'Haul Truck', 62:'Scraper/Tractor',
    63:'Front loader/Bulldozer', 64:'Excavator', 65:'Cement Mixer', 66:'Ground Grader', 71:'Hut/Tent',
    72:'Shed', 73:'Building', 74:'Aircraft Hangar', 76:'Damaged Building', 77:'Facility', 79:'Construction Site',
    83:'Vehicle Lot', 84:'Helipad', 86:'Storage Tank', 89:'Shipping container lot', 91:'Shipping Container',
    93:'Pylon', 94:'Tower'}

old_keys = sorted(list(old_dict.keys()))
new_dict = {old_dict[x]:y for y, x in enumerate(old_keys)}
class_map_dict = {y:old_dict[x] for y, x in enumerate(old_keys)}
with open(output_json_map_path, "w") as json_file:
    json.dump(class_map_dict, json_file)
print(class_map_dict)

{0: 'Fixed-wing Aircraft', 1: 'Small Aircraft', 2: 'Passenger/Cargo Plane', 3: 'Helicopter', 4: 'Passenger Vehicle', 5: 'Small Car', 6: 'Bus', 7: 'Pickup Truck', 8: 'Utility Truck', 9: 'Truck', 10: 'Cargo Truck', 11: 'Truck Tractor w/ Box Trailer', 12: 'Truck Tractor', 13: 'Trailer', 14: 'Truck Tractor w/ Flatbed Trailer', 15: 'Truck Tractor w/ Liquid Tank', 16: 'Crane Truck', 17: 'Railway Vehicle', 18: 'Passenger Car', 19: 'Cargo/Container Car', 20: 'Flat Car', 21: 'Tank car', 22: 'Locomotive', 23: 'Maritime Vessel', 24: 'Motorboat', 25: 'Sailboat', 26: 'Tugboat', 27: 'Barge', 28: 'Fishing Vessel', 29: 'Ferry', 30: 'Yacht', 31: 'Container Ship', 32: 'Oil Tanker', 33: 'Engineering Vehicle', 34: 'Tower crane', 35: 'Container Crane', 36: 'Reach Stacker', 37: 'Straddle Carrier', 38: 'Mobile Crane', 39: 'Dump Truck', 40: 'Haul Truck', 41: 'Scraper/Tractor', 42: 'Front loader/Bulldozer', 43: 'Excavator', 44: 'Cement Mixer', 45: 'Ground Grader', 46: 'Hut/Tent', 47: 'Shed', 48: 'Building', 49

In [21]:
df['TYPE_ID'] = df['TYPE_ID'].apply(lambda x: new_dict[old_dict[x]])
df.head()

,IMAGE_ID,TYPE_ID,X_MIN,Y_MIN,X_MAX,Y_MAX,LONG,LAT
0,2355.tif,48,2712,1145,2746,1177,-90.531640,14.566091
1,2355.tif,48,2720,2233,2760,2288,-90.531603,14.562313
2,2355.tif,48,2687,1338,2740,1399,-90.531694,14.565379
3,2355.tif,48,2691,1201,2730,1268,-90.531704,14.565838
4,2355.tif,48,2671,838,2714,869,-90.531766,14.567149


In [ ]:
# To verify that the data is loaded correctly, let's visualize some images

all_classes = list(class_map_dict.keys())

boxes = get_boxes(df, all_classes)
images_for_display = random.choices(list(boxes.keys()), k=2)
display_images(images_for_display, boxes, image_folder_path, max_images=2, no_cols=1, text=True, class_map=class_map_dict)

## Chunk processing

- Split large TIFF files into chunks
- Save these chunks as JPG files
- Check if the chunks contain annotations
- Reformat the annotations into YOLO format: x_center, y_center, width, height (all normalized)
- Write all YOLO annotations into a dictionary, using the file name as the key

In [23]:
boxes_dict = get_boxes(df)

  0%|          | 0/846 [00:00<?, ?it/s]

In [24]:
def process_image(image_filename, 
                  dir_path=image_folder_path, 
                  boxes=boxes_dict, 
                  out_dir=save_images_folder_path, 
                  chunk_height=CHUNK_HEIGHT, 
                  chunk_width=CHUNK_WIDTH,  
                  jpg_quality=JPEG_COMPRESSION,
                  min_height=MIN_CHUNK_HEIGHT,
                  min_width=MIN_CHUNK_WIDTH,
                  writing=IMAGE_WRITING
                 ):
    """
    Splits an image into small chunks, saves the chunks in JPEG format, and converts the annotations to YOLO format.

    Args:
        image_filename (str): Name of the image file to process.
        dir_path (Path): Path to the directory containing the input images.
        boxes (dict): Dictionary mapping each image to a list of annotations.
        out_dir (Path): Path to the directory where the generated chunks will be saved.
        chunk_height (int): Height of the chunks to create.
        chunk_width (int): Width of the chunks to create.
        jpg_quality (int): JPEG compression quality (value from 0 to 100).
        min_height (int): Minimum height to consider a chunk valid.
        min_width (int): Minimum width to consider a chunk valid.
        writing (bool): If True, saves the generated chunks as JPEG files.

    Returns:
        tuple: A tuple containing:
            - f_names (list): List of the generated chunk file names.
            - widths (list): List of the widths of the generated chunks.
            - heights (list): List of the heights of the generated chunks.
            - y_boxes (dict): Dictionary mapping chunk files to YOLO format annotations.
    """
    
    labels_list = boxes[image_filename]
    image_path = str(dir_path / image_filename)
    image = load_bgr_image(image_path)
    full_height, full_width, _ = image.shape
    y_boxes= {}
    file_names, widths, heights = [], [], []
    
    # row : horizontal position of the chunk (height)
    # col : vertical position of the chunk (width)
    for row in range(0, full_height, chunk_height):
        for col in range(0, full_width, chunk_width):
            stem = image_filename.split('.')[0]
            filenames = str(f"img_{stem}_{row}_{col}.jpg")
            out_pth = str(out_dir / filenames)
            width = chunk_width
            height = chunk_height
            if row + height > full_height:
                height = full_height - row
            if col + width > full_width:
                width = full_width - col
            big_enough = (height >= min_height) and (width >= min_width)
            if big_enough:
                if writing:
                    cv2.imwrite(out_pth, image[row:row+height, col:col+width,:],  [int(cv2.IMWRITE_JPEG_QUALITY), jpg_quality])
                # Find any boxes occurring in the chunk, and convert to YOLO format
                chunk_limits = [col, row, width, height]
                y_boxes[filenames] = match_boxes(labels_list, chunk_limits)
                file_names.append(filenames)
                widths.append(width)
                heights.append(height)
    return file_names, widths, heights, y_boxes

In [25]:
image_filenames = df.IMAGE_ID.unique().tolist()
if DEBUG:
    image_filenames = image_filenames[:len(image_filenames)//120]
    df = df[df['IMAGE_ID'].isin(image_filenames)]

In [26]:
start_time = time.time()
num_threads = mp.cpu_count() 
overall_progress = tqdm_notebook(total=len(image_filenames), desc="Creating and saving image tiles")
yolo_boxes= {}
file_names, widths, heights = [], [], []

with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
    for f_names, c_widths, c_heights, y_boxes in executor.map(process_image, image_filenames):
        file_names.extend(f_names)
        widths.extend(c_widths)
        heights.extend(c_heights)
        yolo_boxes.update(y_boxes)
        overall_progress.update(1)
overall_progress.close()

image_data = {file_names[i]: [widths[i], heights[i]] for i in range(len(file_names))}
time_taken = time.time() - start_time

Creating and saving image tiles:   0%|          | 0/846 [00:00<?, ?it/s]

## YOLO text files

In [27]:
# Iterate through the dictionary, creating a text file for each image with the format: class x y width height, then save the completed text file in the same location, with the same base name as the image.

all_image_files = os.listdir(save_images_folder_path)
for image_filename in tqdm_notebook(all_image_files):
    stem = image_filename.split('.')[0]
    file_name = str (stem) + '.txt'
    txt_path = str(save_images_folder_path / file_name)
    separator = ' '
    with open(txt_path, 'a') as f:
        if image_filename in yolo_boxes:
            for bbox in yolo_boxes[image_filename]:
                txt = separator.join(bbox) + '\n'
                f.write(txt)

  0%|          | 0/72477 [00:00<?, ?it/s]

In [28]:
txt_files = [file for file in os.listdir(save_images_folder_path) if file.endswith('.txt')]
num_txt_files = len(txt_files)
print(f"Number of .txt files: {num_txt_files}")

Number of .txt files: 72477


In [29]:
images_with_boxes = [image_filename for image_filename in all_image_files if image_filename in yolo_boxes]
print(f"Number of images with data in yolo_boxes: {len(images_with_boxes)}")

Number of images with data in yolo_boxes: 72477


In [30]:
text_paths = [save_images_folder_path / x for x in os.listdir(save_images_folder_path) if x.endswith(".txt")]
column_names = ['Class_ID', 'x_center', 'y_center', 'width', 'height']
data = []
for file_path in text_paths:
    with open(file_path, 'r') as file:
        for line in file:
            values = line.strip().split(' ')
            row_data = {col: val for col, val in zip(column_names, values)}
            row_data['File_Name'] = file_path.name
            data.append(row_data)

out_df = pd.DataFrame(data)
out_df['Class_ID']=out_df['Class_ID'].astype(int)
out_df['Class_Name'] = out_df['Class_ID'].map(class_map_dict).fillna('unknown')
out_df = out_df[['File_Name', 'Class_Name', 'Class_ID', 'x_center', 'y_center', 'width', 'height']]
out_df.to_parquet(output_data_parquet_path, index=False)
out_df.head()

,File_Name,Class_Name,Class_ID,x_center,y_center,width,height
0,img_100_0_0.txt,Building,48,0.15,0.02344,0.3,0.04688
1,img_100_0_0.txt,Building,48,0.1,0.25781,0.2,0.30312
2,img_100_0_0.txt,Building,48,0.13438,0.54062,0.2,0.1875
3,img_100_0_0.txt,Building,48,0.24844,0.15625,0.20938,0.3125
4,img_100_0_0.txt,Building,48,0.29688,0.98906,0.0625,0.02187


## Empty images

We try to remove empty images / images without labels

An image is considered "empty" in the context of object detection datasets when its corresponding annotation file (in this case, a .txt file) contains no valid bounding box annotations. This means that the .txt file is either completely empty or contains only whitespace characters (spaces, tabs, newlines) without any actual annotation data.

In [31]:
def remove_empty(image_folder, yolo_boxes, image_data):
    """
    Remove 66% of the empty annotation files (.txt) and their corresponding images.
    Also updates the yolo_boxes and image_data dictionaries to remove the corresponding entries.
        
    Args:
        image_folder (str): Path to the folder containing images and .txt files.
        yolo_boxes (dict): Dictionary with YOLO annotations.
        image_data (dict): Dictionary with image metadata.
        file_names (list): List of image file names.
        widths (list): List of image widths.
        heights (list): List of image heights.
    
    Returns:
        tuple: Updated dictionaries (yolo_boxes, image_data, file_names, widths, heights).
    """
    all_image_files = set(os.listdir(image_folder))  # Set for faster lookups
    empty_files = [] 

    # Check .txt files for empty annotations
    for txt_file in all_image_files:
        if txt_file.endswith('.txt'):
            txt_path = os.path.join(image_folder, txt_file)
            # Check if the file is empty or contains only whitespace
            with open(txt_path, 'r') as file:
                content = file.read().strip()
            if not content:  
                # Determine the corresponding image file
                image_file = txt_file.replace('.txt', '.jpg')  
                empty_files.append(image_file)  # Add image to the list of empty files

    # Select 66% of the empty files to remove
    num_to_remove = int(len(empty_files) * 0.66)
    files_to_remove = random.sample(empty_files, num_to_remove)

    # Remove the .txt files and corresponding images
    for image_file in files_to_remove:
        txt_file = image_file.replace('.jpg', '.txt')  
        txt_path = os.path.join(image_folder, txt_file)
        image_path = os.path.join(image_folder, image_file)

        if os.path.exists(txt_path):
            os.remove(txt_path)
        if os.path.exists(image_path):
            os.remove(image_path)

    # Update yolo_boxes by removing the deleted files
    yolo_boxes = {key: value for key, value in yolo_boxes.items() if key not in files_to_remove}

    # Update image_data by removing the deleted files
    image_data = {key: value for key, value in image_data.items() if key not in files_to_remove}

    # Filter the lists to exclude the removed files
    filtered_file_names = [name for name in file_names if name not in files_to_remove]
    filtered_widths = [widths[i] for i in range(len(file_names)) if file_names[i] not in files_to_remove]
    filtered_heights = [heights[i] for i in range(len(file_names)) if file_names[i] not in files_to_remove]

    return yolo_boxes, image_data, filtered_file_names, filtered_widths, filtered_heights

In [32]:
yolo_boxes, image_data, filtered_file_names, filtered_widths, filtered_heights = remove_empty(save_images_folder_path, yolo_boxes, image_data)


In [33]:
# Verify the number of .txt files in the folder
txt_files = [f for f in os.listdir(save_images_folder_path) if f.endswith('.txt')]
print(f"Number of .txt files in the folder: {len(txt_files)}")

# Count the number of image files (e.g., .jpg files)
image_files = [f for f in os.listdir(save_images_folder_path) if f.endswith('.jpg')]
num_images = len(image_files)
print(f"Number of images in the folder: {num_images}")
images_with_boxes = [image_fn for image_fn in all_image_files if image_fn in yolo_boxes]
print(f"Number of images with data in yolo_boxes: {len(images_with_boxes)}")

Number of .txt files in the folder: 45891
Number of images in the folder: 45891
Number of images with data in yolo_boxes: 45891


In [34]:
from itertools import islice

# Print the first 5 entries of the image_data dictionary to verify the remaining files and their dimensions
for key, value in islice(image_data.items(), 5):
    print(f"{key}: {value}")

img_2355_0_960.jpg: [320, 320]
img_2355_0_1280.jpg: [320, 320]
img_2355_0_1600.jpg: [320, 320]
img_2355_0_1920.jpg: [320, 320]
img_2355_0_2240.jpg: [320, 320]


In [35]:
num_file_names = len(image_data) # The number of keys corresponds to the number of file_names (for now)
num_widths = len([value[0] for value in image_data.values()])  # Count all widths
num_heights = len([value[1] for value in image_data.values()])  # Count all heights

print(f"The number of unique images is: {num_file_names}")
print(f"The number of recorded widths is: {num_widths}")
print(f"The number of recorded heights is: {num_heights}")

The number of unique images is: 45891
The number of recorded widths is: 45891
The number of recorded heights is: 45891


In [ ]:
filenames = [x for x in os.listdir(save_images_folder_path) if x.endswith(".jpg")]
image_list = random.choices(filenames, k=4)
display_yolo_images(image_list, save_images_folder_path, max_images=4, no_cols=1, text=True,  class_map=class_map_dict)

In [ ]:
#without text labels
filenames = [x for x in os.listdir(save_images_folder_path) if x.endswith(".jpg")]
image_list = random.choices(filenames, k=4)
display_yolo_images(image_list, save_images_folder_path, max_images=4, no_cols=2, text=False)

In [ ]:
# Define the specific images to display
specific_images = ['img_1831_0_2240.jpg', 'img_1964_0_1920.jpg', 'img_91_1920_0.jpg', 'img_1284_640_1920.jpg', 'img_128_640_960.jpg']

# Filter the list of images to include only the specific ones
image_list = [x for x in os.listdir(save_images_folder_path) if x in specific_images]

# Display the specific images without text labels
display_yolo_images(image_list, save_images_folder_path, max_images=4, no_cols=2, text=False)

In [39]:

txt_fnames = [save_images_folder_path / x for x in os.listdir(save_images_folder_path) if x.endswith(".txt")]
text_list = random.choices(txt_fnames, k=2)
print(text_list)
for text_f in text_list:
    print(f'Reading {text_f}')
    print_first_n_lines(text_f, 3)  

[WindowsPath('../data/processed/images/img_562_1920_320.txt'), WindowsPath('../data/processed/images/img_494_1600_640.txt')]
Reading ..\data\processed\images\img_562_1920_320.txt
5 0.38438 0.03281 0.0625 0.05312
5 0.68281 0.11406 0.07188 0.05937
5 0.81719 0.12031 0.06562 0.04687
Reading ..\data\processed\images\img_494_1600_640.txt


In [40]:
print(text_list)

[WindowsPath('../data/processed/images/img_562_1920_320.txt'), WindowsPath('../data/processed/images/img_494_1600_640.txt')]


In [41]:
out_data = pd.read_parquet(output_data_parquet_path)
out_data.head()

,File_Name,Class_Name,Class_ID,x_center,y_center,width,height
0,img_100_0_0.txt,Building,48,0.15,0.02344,0.3,0.04688
1,img_100_0_0.txt,Building,48,0.1,0.25781,0.2,0.30312
2,img_100_0_0.txt,Building,48,0.13438,0.54062,0.2,0.1875
3,img_100_0_0.txt,Building,48,0.24844,0.15625,0.20938,0.3125
4,img_100_0_0.txt,Building,48,0.29688,0.98906,0.0625,0.02187


In [42]:
# Count the number of unique images
num_unique_images = out_data['File_Name'].nunique()
print(f"The number of unique images is: {num_unique_images}")

The number of unique images is: 32194


In [43]:
with open(output_json_map_path, "r") as json_file:
    loaded_dict = json.load(json_file)
print(loaded_dict)

{'0': 'Fixed-wing Aircraft', '1': 'Small Aircraft', '2': 'Passenger/Cargo Plane', '3': 'Helicopter', '4': 'Passenger Vehicle', '5': 'Small Car', '6': 'Bus', '7': 'Pickup Truck', '8': 'Utility Truck', '9': 'Truck', '10': 'Cargo Truck', '11': 'Truck Tractor w/ Box Trailer', '12': 'Truck Tractor', '13': 'Trailer', '14': 'Truck Tractor w/ Flatbed Trailer', '15': 'Truck Tractor w/ Liquid Tank', '16': 'Crane Truck', '17': 'Railway Vehicle', '18': 'Passenger Car', '19': 'Cargo/Container Car', '20': 'Flat Car', '21': 'Tank car', '22': 'Locomotive', '23': 'Maritime Vessel', '24': 'Motorboat', '25': 'Sailboat', '26': 'Tugboat', '27': 'Barge', '28': 'Fishing Vessel', '29': 'Ferry', '30': 'Yacht', '31': 'Container Ship', '32': 'Oil Tanker', '33': 'Engineering Vehicle', '34': 'Tower crane', '35': 'Container Crane', '36': 'Reach Stacker', '37': 'Straddle Carrier', '38': 'Mobile Crane', '39': 'Dump Truck', '40': 'Haul Truck', '41': 'Scraper/Tractor', '42': 'Front loader/Bulldozer', '43': 'Excavator',

## YOLO to COCO

The COCO format is explained here. At a high level, we mainly need these three objects:

- images:
{"id": int, "width": int, "height": int, "file_name": str, }
- annotations:
{"id": int, "image_id": int, "category_id": int, "area": float, "bbox": [x, y, width, height]}
- categories:
[{"id": int, "name": str}]

Let's copy the YOLO DataFrame, to extract the image widths and create the BBox category

In [44]:
num_file_names = len(image_data) # The number of keys corresponds to the number of file_names (for now)
num_widths = len([value[0] for value in image_data.values()])  # Count all widths
num_heights = len([value[1] for value in image_data.values()])  # Count all heights

print(f"The number of unique images is: {num_file_names}")
print(f"The number of recorded widths is: {num_widths}")
print(f"The number of recorded heights is: {num_heights}")

The number of unique images is: 45891
The number of recorded widths is: 45891
The number of recorded heights is: 45891


In [45]:
image_data = {'width': filtered_widths, 'height': filtered_heights, 'file_name': filtered_file_names} # Ricreo il dizionario con 3 chiavi ma utilizzando le liste filtraten dopo l'eliminazione fatta prima
im_df = pd.DataFrame(image_data)
im_df['id'] = im_df['file_name'].str.replace(r'\D', '', regex=True).astype(int)
im_df.head()

,width,height,file_name,id
0,320,320,img_2355_0_960.jpg,23550960
1,320,320,img_2355_0_1280.jpg,235501280
2,320,320,img_2355_0_1600.jpg,235501600
3,320,320,img_2355_0_1920.jpg,235501920
4,320,320,img_2355_0_2240.jpg,235502240


In [46]:
num_file_names = len(image_data['file_name'])  # Number of remaining files
num_widths = len(image_data['width'])         # Number of remaining widths
num_heights = len(image_data['height'])       # Number of remaining heights

print(f"The number of unique images is: {num_file_names}")
print(f"The number of recorded widths is: {num_widths}")
print(f"The number of recorded heights is: {num_heights}")

The number of unique images is: 45891
The number of recorded widths is: 45891
The number of recorded heights is: 45891


In [47]:
num_images_in_im_df = len(im_df)
print(f"The number of images in im_df is: {num_images_in_im_df}")

The number of images in im_df is: 45891


In [48]:
def row_to_dict(row):
    return {
        'id': row['id'],
        'width': row['width'],
        'height':row['height'],
        'file_name':row['file_name']
    }

im_list = im_df.apply(lambda row: row_to_dict(row), axis=1).tolist()
[print(val) for val in im_list[:4]]

{'id': 23550960, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_960.jpg'}
{'id': 235501280, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_1280.jpg'}
{'id': 235501600, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_1600.jpg'}
{'id': 235501920, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_1920.jpg'}


[None, None, None, None]

In [49]:
# Merge the images dataframe with the annotations to work out the absolute pixel values, plus a bit more re-organising

annotations_df = out_data.copy()
annotations_df['image_id'] = annotations_df['File_Name'].str.replace(r'\D', '', regex=True).astype(int)
annotations_df= annotations_df.rename(columns={'height': 'h', 'width': 'w'})

an_df = annotations_df.merge(im_df, left_on='image_id', right_on='id', how='left')
an_df['x_center']= (an_df['x_center'].astype(np.float64)*an_df['width']).round(decimals=0)
an_df['y_center']= (an_df['y_center'].astype(np.float64)*an_df['height']).round(decimals=0)
an_df['w']= (an_df['w'].astype(np.float64)*an_df['width']).round(decimals=0)
an_df['h']= (an_df['h'].astype(np.float64)*an_df['height']).round(decimals=0)
an_df['Class_ID']= an_df['Class_ID'].astype(int)
an_df = an_df.drop(columns=['File_Name', 'file_name', 'width', 'height', 'id'])
an_df['left'] = (an_df['x_center'] - an_df['w']/2).round(decimals=0)
an_df['top'] =  (an_df['y_center'] - an_df['h']/2).round(decimals=0)
an_df['bbox'] = ('[' + an_df['left'].astype(str) + ', ' 
              + an_df['top'].astype(str) + ', ' 
              + an_df['w'].astype(str) + ', '
              + an_df['h'].astype(str) + ']')
an_df['area'] = an_df['w'] * an_df['h']
an_df = an_df.drop(columns=['x_center', 'y_center', 'w', 'h', 'left', 'top', 'Class_Name'])
an_df.reset_index(inplace=True)
an_df.rename(columns={'index': 'id'}, inplace=True)
an_df.head()

,id,Class_ID,image_id,bbox,area
0,0,48,10000,"[0.0, 0.0, 96.0, 15.0]",1440.0
1,1,48,10000,"[0.0, 34.0, 64.0, 97.0]",6208.0
2,2,48,10000,"[11.0, 143.0, 64.0, 60.0]",3840.0
3,3,48,10000,"[46.0, 0.0, 67.0, 100.0]",6700.0
4,4,48,10000,"[85.0, 312.0, 20.0, 7.0]",140.0


In [50]:
# Convert the DataFrame to a list of dictionaries, where each dictionary represents a row in the DataFrame with the specified keys.

def row_to_dict(row):
    return {
        'id': row['id'],
        'image_id' : row['image_id'],
        'category_id': row['Class_ID'],
        'area':row['area'],
        'bbox':row['bbox']
    }

an_list = an_df.apply(lambda row: row_to_dict(row), axis=1).tolist()
print(an_list[:4])

[{'id': 0, 'image_id': 10000, 'category_id': 48, 'area': 1440.0, 'bbox': '[0.0, 0.0, 96.0, 15.0]'}, {'id': 1, 'image_id': 10000, 'category_id': 48, 'area': 6208.0, 'bbox': '[0.0, 34.0, 64.0, 97.0]'}, {'id': 2, 'image_id': 10000, 'category_id': 48, 'area': 3840.0, 'bbox': '[11.0, 143.0, 64.0, 60.0]'}, {'id': 3, 'image_id': 10000, 'category_id': 48, 'area': 6700.0, 'bbox': '[46.0, 0.0, 67.0, 100.0]'}]


In [51]:
# Convert the class map dictionary to a list of dictionaries, where each dictionary has a single key-value pair corresponding to the class ID and class name.

cat_list = [{key:val} for key,val in class_map_dict.items()]
print(cat_list[:4])

[{0: 'Fixed-wing Aircraft'}, {1: 'Small Aircraft'}, {2: 'Passenger/Cargo Plane'}, {3: 'Helicopter'}]


In [52]:
# Create a dictionary with three keys: 'images', 'annotations', and 'categories'. The value of each key is the corresponding list of dictionaries created in the previous steps. Then, save this dictionary as a JSON file.

out_json_data = {'images': im_list, 'annotations': an_list, 'categories': cat_list}
with open(coco_json_path, 'w') as json_file:
    json.dump(out_json_data, json_file, indent=4)
    
for key, value in out_json_data.items():
    print(key, value[:5])

images [{'id': 23550960, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_960.jpg'}, {'id': 235501280, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_1280.jpg'}, {'id': 235501600, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_1600.jpg'}, {'id': 235501920, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_1920.jpg'}, {'id': 235502240, 'width': 320, 'height': 320, 'file_name': 'img_2355_0_2240.jpg'}]
annotations [{'id': 0, 'image_id': 10000, 'category_id': 48, 'area': 1440.0, 'bbox': '[0.0, 0.0, 96.0, 15.0]'}, {'id': 1, 'image_id': 10000, 'category_id': 48, 'area': 6208.0, 'bbox': '[0.0, 34.0, 64.0, 97.0]'}, {'id': 2, 'image_id': 10000, 'category_id': 48, 'area': 3840.0, 'bbox': '[11.0, 143.0, 64.0, 60.0]'}, {'id': 3, 'image_id': 10000, 'category_id': 48, 'area': 6700.0, 'bbox': '[46.0, 0.0, 67.0, 100.0]'}, {'id': 4, 'image_id': 10000, 'category_id': 48, 'area': 140.0, 'bbox': '[85.0, 312.0, 20.0, 7.0]'}]
categories [{0: 'Fixed-wing Aircraft'}, {1: 'Small Aircraft

## Check image size

In [53]:
def check_image_sizes(directory_path):
    size_counts = defaultdict(int)

    # Obtain the list of files in the directory
    files = [f for f in os.listdir(directory_path) if f.endswith(('.jpg'))]

    # Add a progress bar for iterating over the files
    for filename in tqdm(files, desc="Processing images"):
        img_path = os.path.join(directory_path, filename)
        with Image.open(img_path) as img:
            size = img.size  # (width, height)
            size_counts[size] += 1

    # Create groups for the required sizes
    size_groups = {
        'Smaller than 320x320': [],
        '320x320': [],
        'Larger than 320x320': [],
    }

    # Add the sizes to the appropriate groups
    for size, count in size_counts.items():
        width, height = size
        if width < 320 and height < 320:
            size_groups['Smaller than 320x320'].append((size, count))
        elif width == 320 and height == 320:
            size_groups['320x320'].append((size, count))
        elif width > 320 and height > 320:
            size_groups['Larger than 320x320'].append((size, count))

    # Sort the sizes by area (width * height) in descending order
    for group, items in size_groups.items():
        sorted_items = sorted(items, key=lambda x: x[0][0] * x[0][1], reverse=True)  # ordina per area
        size_groups[group] = sorted_items

    # Print the groups with the number of images per size and the total for each range
    for group, items in size_groups.items():
        total = sum(count for _, count in items)
        print(f"{group} (Total: {total} images):")
        for size, count in items:
            print(f"  Size {size}: {count} images")
        print()

# Run the function with the directory path
check_image_sizes(save_images_folder_path)

Processing images: 100%|██████████| 45891/45891 [14:13<00:00, 53.76it/s] 

Smaller than 320x320 (Total: 0 images):

320x320 (Total: 45891 images):
  Size (320, 320): 45891 images

Larger than 320x320 (Total: 0 images):



In [54]:
# Open the generated COCO JSON file and load the data to verify its structure and content.

with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

In [55]:
def check_image_size(image, img_id, coco_data):
    # Find the image dimensions in the JSON
    image_info = next((img for img in coco_data['images'] if img['id'] == img_id), None)
    if image_info:
        width, height = image_info['width'], image_info['height']
        img_width, img_height = image.size
        # Raise an exception only if the dimensions do not match
        assert img_width == width and img_height == height, (
            f"Incorrect dimensions: JSON ({width}, {height}), image ({img_width}, {img_height})"
        )
    else:
        raise ValueError(f"Image with ID {img_id} not found in JSON.")

def check_all_images(folder_path, coco_data):
    folder_path = Path(folder_path)
    errors_found = False  # Flag to keep track of errors
    check_count = 0  # Count the number of images checked
    
    # Iterate through all files in the folder
    for image_path in folder_path.iterdir():
        if image_path.is_file() and image_path.suffix in ['.jpg']:  # Check only image files
            check_count += 1  # Increment the image counter
            # Find the corresponding ID based on the file name
            img_id = int(''.join(filter(str.isdigit, image_path.stem)))  # Extract numbers from the name
            try:
                with Image.open(image_path) as img:
                    check_image_size(img, img_id, coco_data)
            except (AssertionError, ValueError) as e:
                errors_found = True
                print(f"Error for image {image_path.name}: {e}")
            except Exception as e:
                errors_found = True
                print(f"Generic error for image {image_path.name}: {e}")

    # Print the final result
    if not errors_found:
        print(f"Check completed, no errors found. Number of images checked: {check_count}")
    else:
        print(f"Check completed with errors. Number of images checked: {check_count}")

# Path to the image directory
check_all_images(save_images_folder_path, coco_data)

Check completed, no errors found. Number of images checked: 45891


## Check labels

In [56]:
def check_empty_txt_files(directory_path):
    # Obtain all .txt files in the directory
    txt_files = [f for f in os.listdir(directory_path) if f.endswith(".txt")]
    
    empty_files_count = 0
    total_lines = 0
    non_empty_files_count = 0
    
    # Check if each file is empty
    for txt_file in txt_files:
        file_path = os.path.join(directory_path, txt_file)
        if os.path.getsize(file_path) == 0:
            empty_files_count += 1
        else:
            non_empty_files_count += 1
            with open(file_path, 'r') as f:
                lines = f.readlines()
                total_lines += len(lines)
    
    # Calculate the average number of lines for non-empty files
    avg_lines = total_lines / non_empty_files_count if non_empty_files_count > 0 else 0
    
    # Print the number of empty files and the average number of lines in non-empty files
    print(f"Number of empty .txt files: {empty_files_count}")
    print(f"Number of non-empty .txt files: {non_empty_files_count}")
    print(f"Average number of lines per non-empty file: {avg_lines:.2f}")

check_empty_txt_files(save_images_folder_path)

Number of empty .txt files: 13697
Number of non-empty .txt files: 32194
Average number of lines per non-empty file: 20.75


## Category distribution

In [ ]:
with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

# Create a dictionary to map category_id to category name
category_mapping = {str(index): list(category.values())[0] for index, category in enumerate(coco_data['categories'])}

# Extract category_id from annotations
category_ids = [annotation['category_id'] for annotation in coco_data['annotations']]

# Count the occurrences of each category_id
category_counts = Counter(category_ids)

# Category groups
category_groups = {
    "Aircraft": [0, 1, 2, 3],
    "Passenger Vehicle": [4, 5, 6, 53],
    "Truck": [7, 8, 9, 10, 11, 12, 13, 14, 15],
    "Railway Vehicle": [17, 18, 19, 20, 21, 22],
    "Maritime Vessel": [23, 24, 25, 26, 27, 28, 29, 30, 31, 32],
    "Engineering Vehicle": [16, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45],
    "Building": [46, 47, 48, 49, 50, 51, 52, 59],
    "Helipad": [54],
    "Storage Tank": [55],
    "Shipping Container": [56, 57],
    "Pylon": [58],
}

# Create a dictionary for the aggregated categories with their occurrences
grouped_occurrences = {}
for group_name, category_ids in category_groups.items():
    # Sum the occurrences for the categories in each group
    grouped_occurrences[group_name] = sum(category_counts[cat_id] for cat_id in category_ids)

# Sort the grouped categories by occurrences in descending order
grouped_occurrences = dict(sorted(grouped_occurrences.items(), key=lambda item: item[1], reverse=True))

# Prepare the data for the chart
groups = list(grouped_occurrences.keys())
occurrences = list(grouped_occurrences.values())

# Create the bar chart
plt.figure(figsize=(10, 6))
plt.barh(groups, occurrences, color='skyblue')
plt.xlabel('Number of Occurrences')
plt.ylabel('Category Groups')
plt.title('Distribution of Occurrences by Category Group')
plt.tight_layout()

# Show the chart
plt.show()

# Print the occurrences for each group and their respective IDs
sorted_groups = sorted(category_groups.items(), key=lambda item: sum(category_counts.get(cat_id, 0) for cat_id in item[1]))

for group, category_ids in sorted_groups:
    # Calculate the total number of occurrences for the group
    total_count = sum(category_counts.get(cat_id, 0) for cat_id in category_ids)
    
    print(f"Group: {group}, Total Occurrences: {total_count}")
    
    for cat_id in category_ids:
        category_name = category_mapping[str(cat_id)]
        count = category_counts.get(cat_id, 0)
        print(f"  category_id: {cat_id}, Category: {category_name}, Occurrences: {count}")
    
    print("\n")

In [58]:
new_coco_json_path = output_dataset_path / 'coco_annotations_new.json'

In [59]:
# Load the original COCO file
with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

# Create new categories in the required format
new_categories = [{str(idx): group} for idx, group in enumerate(category_groups.keys())]

# Create a reverse mapping of groups (old category_id -> new group_id)
group_mapping = {}
for group_id, (group_name, category_ids) in enumerate(category_groups.items(), start=0):
    for cat_id in category_ids:
        group_mapping[cat_id] = group_id

# Update the category_id in the annotations
for annotation in coco_data["annotations"]:
    original_id = annotation["category_id"]
    annotation["category_id"] = group_mapping.get(original_id, -1)  # Use -1 for any unmapped category_id

# Check for unmapped annotations
unmapped = [ann for ann in coco_data["annotations"] if ann["category_id"] == -1]
if unmapped:
    print(f"Warning: {len(unmapped)} annotations with unmapped category_id.")

# Update the categories dictionary
coco_data["categories"] = new_categories

# Save the modified COCO file
new_coco_json_path = Path(output_dataset_path) / 'coco_annotations_new.json'
with open(new_coco_json_path, "w") as f:
    json.dump(coco_data, f, indent=4)

print(f"Modified COCO file saved at {new_coco_json_path}.")

Modified COCO file saved at ..\data\processed\coco_annotations_new.json.


In [ ]:
# Load the modified COCO file
with open(new_coco_json_path, 'r') as f:
    coco_data = json.load(f)

# Create a dictionary to map category_id to category name from the new format
category_mapping = {
    int(list(category.keys())[0]): list(category.values())[0]
    for category in coco_data["categories"]
}

# Extract category_id from annotations
category_ids = [annotation["category_id"] for annotation in coco_data["annotations"]]

# Count the occurrences of each category_id
category_counts = Counter(category_ids)

# Map category names to their occurrences
category_occurrences = {
    cat_id: (category_mapping[cat_id], count)
    for cat_id, count in category_counts.items()
    if cat_id in category_mapping
}

# Sort categories by occurrences in descending order
category_occurrences = dict(sorted(category_occurrences.items(), key=lambda item: item[1][1], reverse=True))

# Prepare data for the plot
categories = [f"{cat_id} - {name}" for cat_id, (name, _) in category_occurrences.items()]
occurrences = [count for _, (_, count) in category_occurrences.items()]

# Create the bar plot
plt.figure(figsize=(10, 8))
plt.barh(categories, occurrences, color="skyblue")
plt.xlabel("Number of Occurrences")
plt.ylabel("Categories")
plt.title("Distribution of Occurrences by Category")
plt.tight_layout()

# Show the plot
plt.show()

# Print the occurrences for each category with ID
for cat_id, (name, count) in category_occurrences.items():
    print(f"Category ID: {cat_id}, Name: {name}, Occurrences: {count}")

In [61]:
# Name of the zip file to create
zip_file_name = "our-xview-dataset.zip"

# List of files and folders to include in the zip
items_to_zip = [
    new_coco_json_path, # The modified COCO JSON file with the new category groups
    save_images_folder_path    # Folder containing the generated images and .txt files
]

# Function to add files and folders to the zip
def zip_folder(zipf, folder_path, base_folder=""):
    for root, _, files in os.walk(folder_path):
        for file in files:
            if not file.endswith(".txt"):  # Exclude .txt files
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, base_folder)
                zipf.write(file_path, arcname)

# Creation of the zip file
import zipfile
with zipfile.ZipFile(zip_file_name, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for item in items_to_zip:
        if os.path.exists(item):  # Check if the file or folder exists
            if os.path.isdir(item):  # If it's a folder, add all its contents
                zip_folder(zipf, item, os.getcwd())
            else:  # If it's a file, add it directly
                zipf.write(item)
        else:
            print(f"Item not found: {item}")